In [ ]:
import numpy as np
import dotenv
dotenv.load_dotenv("exploration/querying/.env")
# %%
import sys
import os

sys.path.insert(0, os.path.abspath("../../"))
sys.path.append(os.path.abspath("../../backend/src"))
sys.path.append(os.path.abspath("backend/src"))

from rdflib.plugins.stores.sparqlstore import SPARQLStore

from backend.src.explorative.explorative_support import GuidanceManager
from backend.src.ontology import OntologyManager, OntologyConfig, Graph
from backend.src.explorative.llm_query import (
    EnrichedEntitiesRelations,
    EnrichedEntity,
    EnrichedRelation,
    EntitiesRelations,
    LLMQuery,
    QueryProgress,
)
from tqdm import tqdm
import pandas as pd

In [ ]:
from seqeval import metrics
from sklearn.metrics import f1_score
import networkx as nx

In [ ]:
from backend.src.eval_config import get_config

In [ ]:
generated_queries = pd.read_csv("exploration/querying/llama_examples.csv", index_col=0)
generated_queries["erl"] = generated_queries["erl"].apply(
    lambda x: EnrichedEntitiesRelations.model_validate_json(x)
)
generated_queries

In [ ]:
from backend.src.api.config import ManMan

setup = get_config("dbpedia", -1)
setup.agent.max_steps = generated_queries["n_nodes"].max() * 2
man_man = ManMan(
    config_id=id,
    eval_cfg=setup,
)
setup.agent.base_url, setup.device


In [ ]:
from backend.src.agent.graph_tools import QueryGraph


def graph_to_erl(graph: QueryGraph) -> EnrichedEntitiesRelations:
    entities = []
    for subject in graph.subjects:
        print(f"Processing subject {subject.id} with URI {subject.uri}")
        subject_full = subject.get_full(man_man, load_properties=False)
        entities.append(
            EnrichedEntity(
                subject=subject_full,
                identifier=subject.id,
                constraints=[],
                type=subject.uri,
            )
        )
    links = []
    for relation in graph.links:
        print(f"Processing relation {relation.uri} from {relation.from_subject_id} to {relation.to_subject_id}")
        relation_full = relation.get_full(man_man)
        links.append(
            EnrichedRelation(
                link=relation_full,
                relation=relation.uri,
                entity=relation.from_subject_id,
                target=relation.to_subject_id,
            )
        )
    return EnrichedEntitiesRelations(
        entities=entities,
        relations=links,
    )

In [ ]:
for i, row in generated_queries.iterrows():
    erl: EnrichedEntitiesRelations = row["erl"]
    for ent in erl.entities:
        ent.type = man_man.ontology_manager.to_prefix(ent.type)
    query: str = row["response"]
    session = man_man.build_agent_session(f"test_{i}")
    print(
        f"Running query {i} with {len(erl.entities)} nodes and {len(erl.relations)} edges"
    )
    for resp in session.run_turn(
        content=query,
    ):
        if hasattr(resp, "delta") and resp.delta:
            print(resp.delta, end="")
    print(f"Query {i} completed")
    # convert the graph to the erl format and compare with the expected erl
    erl_result = graph_to_erl(session.graph)
    break

In [ ]:
session.agent.tools.graph

In [ ]:
erl_result = graph_to_erl(session.agent.tools.graph)
erl_result

In [ ]:
(
    [[ent.subject.subject_id for ent in erl_result.entities]],
    [
        [
            man_man.ontology_manager.to_prefix(ent.subject.subject_id)
            for ent in erl.entities
        ]
    ],
)

In [ ]:
def graph_from_erl(erl: EnrichedEntitiesRelations):
    G = nx.DiGraph()
    for node in erl.entities:
        G.add_node(node.identifier, id=node.subject.subject_id, type=node.type)
    for link in erl.relations:
        G.add_edge(
            link.entity,
            link.target,
            weight=link.link.instance_count,
            id=link.link.property_id,
        )
    return G

In [ ]:
def f1k(y_true, y_pred, k: int = None):
    rel_set = set(y_true)
    # print(rel_set)
    doc_set = set(y_pred[:k])
    tp = len(doc_set.intersection(rel_set))  # docs that are in both -relevant docs
    fp = len(
        doc_set.difference(rel_set)
    )  # docs that are not in relevant set - irrelevant docs (false positiv)
    fn = len(
        rel_set.difference(doc_set)
    )  # relevant docs that are not present in doc set - missing docs
    if tp == 0:
        return 0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    return 2 * precision * recall / (precision + recall)

In [ ]:

for ent in erl.entities:
    ent.subject.subject_id = man_man.ontology_manager.to_prefix(ent.subject.subject_id)
for rel in erl.relations:
    rel.link.property_id = man_man.ontology_manager.to_prefix(rel.link.property_id)

In [ ]:
f1_score_ents = f1k(
    [ent.subject.subject_id for ent in erl_result.entities],
    [ent.subject.subject_id for ent in erl.entities],
)
f1_score_ents


In [ ]:
target_graph = graph_from_erl(erl)
retrieved_grap = graph_from_erl(erl_result)


def edge_match(a, b):
    return a["id"] == b["id"] if "id" in a and "id" in b else False


def node_match(a, b):
    return a["id"] == b["id"] if "id" in a and "id" in b else False


1 - nx.graph_edit_distance(
    target_graph, retrieved_grap, edge_match=edge_match, node_match=node_match
) / (
    len(target_graph.edges)
    + len(retrieved_grap.edges)
    + len(target_graph.nodes)
    + len(retrieved_grap.nodes)
)

In [ ]:
def run_eval(query: pd.Series, llm_query: LLMQuery):
    target_erl: EnrichedEntitiesRelations = query["erl"]
    query = query["response"]
    progress = QueryProgress(id="0", max_steps=1, start_time="0")
    llm_query.run_query(query=query, progress=progress)
    f1_score_ents = f1k(
        [ent.type for ent in target_erl.entities],
        [ent.type for ent in progress.enriched_relations.entities],
    )
    target_graph = graph_from_erl(target_erl)
    retrieved_grap = graph_from_erl(progress.enriched_relations)
    edit_distance = 1 - nx.graph_edit_distance(
        target_graph, retrieved_grap, edge_match=edge_match, node_match=node_match
    ) / (
        len(target_graph.edges)
        + len(retrieved_grap.edges)
        + len(target_graph.nodes)
        + len(retrieved_grap.nodes)
    )
    return f1_score_ents, edit_distance


def run_evals(queries: pd.DataFrame, llm_query: LLMQuery):
    results = queries.copy()
    for i, query in tqdm(queries.iterrows(), total=len(queries)):
        f1_score_ents, edit_distance = run_eval(query, llm_query)
        results.loc[i, "f1_score"] = f1_score_ents
        results.loc[i, "edit_distance"] = edit_distance
    return results


for zero_shot in [True, False]:
    print(f"oneshot={zero_shot}")
    query_man = LLMQuery(topic_man, zero_shot=zero_shot)
    results = run_evals(generated_queries, llm_query=query_man)
    results.to_csv(
        f"restuls/eval_results_{'oneshot' if zero_shot else 'zeroshot'}_{topic_man.llm_model_id.replace('/', '-')}.csv"
    )